# 01 · Generación del dataset sintético

Genera las cuatro tablas de origen (`tiendas`, `productos`, `clientes`, `transacciones`) que simulan la información de un ERP y un CRM, con imperfecciones de calidad introducidas de forma controlada (informe técnico, apartado 2.1).

**Reproducibilidad.** Las semillas de `numpy` y `random` fijan todos los campos que intervienen en el análisis: identificadores, fechas, importes, cantidades, tiendas, canales y la posición de los nulos. Los campos de texto generados con Faker (nombre y email del cliente, nombre del producto) no están sembrados y cambian entre ejecuciones; no intervienen en ningún cálculo.

Por defecto el notebook **no sobrescribe** los CSV de `data/raw/` si ya existen, para preservar la capa raw publicada.

## 1. Librerías y parámetros

In [1]:
import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

np.random.seed(42)
random.seed(42)
fake = Faker('es_ES')

RAW_PATH = os.path.join('..', '..', 'data', 'raw')
SOBRESCRIBIR = False   # True para regenerar data/raw desde cero

In [2]:
# ── PARÁMETROS ─────────────────────────────────────────────
N_CLIENTES    = 1_000
N_TRANSACC    = 8_000
FECHA_INICIO  = datetime(2023, 1, 1)
FECHA_FIN     = datetime(2025, 12, 31)

CIUDADES = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao',
            'Zaragoza', 'Málaga', 'Alicante']

CANALES_ADQUISICION = ['organico', 'paid', 'referral', 'tienda']

CATEGORIAS = ['Montaña', 'Running', 'Ciclismo', 'Padel']

TIENDAS = [
    ('T01', 'Cumbres Madrid',     'Madrid'),
    ('T02', 'Cumbres Barcelona',  'Barcelona'),
    ('T03', 'Cumbres Valencia',   'Valencia'),
    ('T04', 'Cumbres Sevilla',    'Sevilla'),
    ('T05', 'Cumbres Web',        'Online'),
]


# ── HELPERS ────────────────────────────────────────────────
def fecha_random():
    delta = (FECHA_FIN - FECHA_INICIO).days
    return FECHA_INICIO + timedelta(days=random.randint(0, delta))

## 2. Creación de las tablas

In [3]:
# ── 1. TIENDAS ─────────────────────────────────────────────
tiendas = pd.DataFrame(TIENDAS, columns=['store_id', 'nombre', 'ciudad'])
tiendas

,store_id,nombre,ciudad
0,T01,Cumbres Madrid,Madrid
1,T02,Cumbres Barcelona,Barcelona
2,T03,Cumbres Valencia,Valencia
3,T04,Cumbres Sevilla,Sevilla
4,T05,Cumbres Web,Online


In [4]:
# ── 2. PRODUCTOS ───────────────────────────────────────────
productos = pd.DataFrame([
    {
        'product_id':  f"P{str(i).zfill(3)}",
        'nombre':      f"{fake.word().capitalize()} {cat}",
        'categoria':   cat,
        'precio_base': round(random.choice([
            random.uniform(15, 60),    # accesorios
            random.uniform(60, 180),   # ropa técnica
            random.uniform(180, 500),  # equipamiento
        ]), 2),
    }
    for i, cat in enumerate(
        [c for c in CATEGORIAS for _ in range(12)], start=1
    )
])

In [5]:
# ── 3. CLIENTES ────────────────────────────────────────────
clientes = pd.DataFrame([
    {
        'customer_id':       f"C{str(i).zfill(4)}",
        'nombre':            fake.first_name() + ' ' + fake.last_name(),
        'email':             fake.email(),
        'ciudad':            random.choice(CIUDADES),
        'canal_adquisicion': random.choice(CANALES_ADQUISICION),
        'fecha_alta':        fecha_random().strftime('%Y-%m-%d'),
    }
    for i in range(1, N_CLIENTES + 1)
])

# Suciedad mínima y documentada
clientes.loc[clientes.sample(frac=0.04).index, 'email']  = None  # emails sin registrar
clientes.loc[clientes.sample(frac=0.05).index, 'ciudad'] = None  # ciudad no informada

In [6]:
# ── 4. TRANSACCIONES ───────────────────────────────────────
ids_cliente  = clientes['customer_id'].tolist()
ids_producto = productos['product_id'].tolist()
ids_tienda   = tiendas['store_id'].tolist()

# Distribución realista: algunos clientes compran más que otros
pesos = np.random.exponential(1, N_CLIENTES)
pesos /= pesos.sum()

transacciones = pd.DataFrame([
    {
        'transaction_id': f"TX{str(i).zfill(5)}",
        'customer_id':    np.random.choice(ids_cliente, p=pesos),
        'product_id':     random.choice(ids_producto),
        'store_id':       random.choice(ids_tienda),
        'cantidad':       random.choices([1, 2, 3], weights=[0.70, 0.22, 0.08])[0],
        'importe':        round(random.uniform(20, 400), 2),
        'fecha':          fecha_random().strftime('%Y-%m-%d'),
    }
    for i in range(1, N_TRANSACC + 1)
])

# Suciedad mínima y documentada
transacciones.loc[transacciones.sample(frac=0.02).index, 'importe'] *= 8   # outliers POS
transacciones.loc[transacciones.sample(frac=0.01).index, 'importe'] *= -1  # devoluciones (importe negativo)
transacciones.loc[transacciones.sample(frac=0.03).index, 'store_id'] = None

In [7]:
print("── Cumbres Outdoor · Datos generados ──────────────────")
print(f"  Clientes       : {len(clientes):>5} filas")
print(f"  Transacciones  : {len(transacciones):>5} filas")
print(f"  Productos      : {len(productos):>5} filas")
print(f"  Tiendas        : {len(tiendas):>5} filas")
print()
print("── Suciedad intencional ────────────────────────────────")
for nombre, df in [('clientes', clientes), ('transacciones', transacciones)]:
    nulos = df.isnull().sum().sum()
    print(f"  {nombre:15s}: {nulos} nulos ({nulos/df.size*100:.1f}%)")
print()

── Cumbres Outdoor · Datos generados ──────────────────
  Clientes       :  1000 filas
  Transacciones  :  8000 filas
  Productos      :    48 filas
  Tiendas        :     5 filas

── Suciedad intencional ────────────────────────────────
  clientes       : 90 nulos (1.5%)
  transacciones  : 240 nulos (0.4%)



## 3. Exportación a la capa raw

In [8]:
os.makedirs(RAW_PATH, exist_ok=True)

for nombre, df in [('tiendas', tiendas), ('productos', productos),
                   ('clientes', clientes), ('transacciones', transacciones)]:
    destino = os.path.join(RAW_PATH, f'{nombre}.csv')
    if os.path.exists(destino) and not SOBRESCRIBIR:
        print(f'{nombre}.csv ya existe: no se sobrescribe')
    else:
        df.to_csv(destino, index=False)
        print(f'{nombre}.csv exportado')

tiendas.csv ya existe: no se sobrescribe
productos.csv ya existe: no se sobrescribe
clientes.csv ya existe: no se sobrescribe
transacciones.csv ya existe: no se sobrescribe
